# CamboVerse: train a 3D Gaussian Splat on Kaggle

Photos or video frames (your Kaggle Dataset) → pick & resize frames → COLMAP camera poses → `splatfacto` training → `.ply` → web-ready `.splat` for `public/models/`.

**Before running:**
1. **Add Input** (right panel) → your photo dataset.
2. **Settings → Accelerator = GPU T4 x2** (or P100), **Internet = On** (needs a phone-verified account).
3. Edit the **Config** cell below (site name, iterations, splat budget).
4. For the full run, use **Save Version → Save & Run All (Commit)** so it keeps running if you close the tab (max 12 h).

Full guide: `docs/TRAIN_3DGS_KAGGLE.md` in the CamboVerse repo. Code: Apache-2.0. Keep your photos' CC-BY provenance note with the output.

In [ ]:
# ---- Config ----------------------------------------------------------------
SITE = "num-statue"       # output filename: <SITE>.splat (match the app's public/models/ name)
ITERATIONS = 7000          # 7000 = quick test; 30000 = final quality (~4x longer)
MAX_SPLATS = 150_000       # mobile budget for a ~$150 Android over 4G (docs/CAPTURE.md)
IMAGES = None              # None = auto-detect the folder with the most images under /kaggle/input

# Video frames: neighbours are near-duplicates, so train on an even subset.
TARGET_FRAMES = 300        # 200-400 is the sweet spot; more = much slower COLMAP
LONG_SIDE = 1600           # resize every frame to one size (handles mixed 2.7K / 1080p clips)
MATCHING = "sequential"    # 'sequential' for video frames (fast); 'exhaustive' for loose photos

WORK = "/kaggle/working"
FRAMES = f"{WORK}/frames"  # the prepared subset COLMAP actually sees
PROC = f"{WORK}/proc"      # COLMAP output (cache this as a Dataset to skip SfM next time)
OUT = f"{WORK}/out"
EXPORT = f"{WORK}/export"

In [ ]:
# ---- Find the photos -------------------------------------------------------
import os

IMG_EXT = (".jpg", ".jpeg", ".png")
counts = {}
for root, _, files in os.walk("/kaggle/input"):
    n = sum(f.lower().endswith(IMG_EXT) for f in files)
    if n:
        counts[root] = n
for d, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"{n:5d}  {d}")

assert counts, "No images found under /kaggle/input. Did you 'Add Input' your dataset?"
if IMAGES is None:
    IMAGES = max(counts, key=counts.get)
print("\nUsing:", IMAGES, f"({counts.get(IMAGES, '?')} images)")

In [ ]:
# ---- Pick an even subset of frames, in shooting order, at one size ---------
# Sort by the number in the filename ("status (10).jpg" after "status (9).jpg",
# not after "status (1).jpg") so sequential matching sees neighbours in order.
import re, shutil
import numpy as np
from collections import Counter
from PIL import Image

def frame_key(name):
    nums = re.findall(r"\d+", name)
    return (int(nums[-1]) if nums else 0, name)

files = sorted((f for f in os.listdir(IMAGES) if f.lower().endswith(IMG_EXT)), key=frame_key)
idx = np.linspace(0, len(files) - 1, min(TARGET_FRAMES, len(files))).round().astype(int)
picked = [files[i] for i in idx]

shutil.rmtree(FRAMES, ignore_errors=True)
os.makedirs(FRAMES)
sizes = Counter()
for i, f in enumerate(picked):
    im = Image.open(f"{IMAGES}/{f}").convert("RGB")
    sizes[im.size] += 1
    im.thumbnail((LONG_SIDE, LONG_SIDE), Image.LANCZOS)
    im.save(f"{FRAMES}/frame_{i:05d}.jpg", quality=95)

print(f"{len(files)} images -> {len(picked)} evenly spaced frames in {FRAMES}")
print("source sizes:", dict(sizes))
if len(sizes) > 1:
    print("Mixed resolutions: all resized to one size. If COLMAP registers few frames, "
          "the clips may use different zoom/lens; try keeping only one resolution.")
if len(picked) < 50:
    print("WARNING: under ~50 frames usually gives a poor reconstruction.")

In [ ]:
# ---- Check the GPU ---------------------------------------------------------
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator = GPU T4 x2"
TORCH_BEFORE = torch.__version__
print("torch", torch.__version__, "| CUDA", torch.version.cuda)

In [ ]:
# ---- Install COLMAP + Nerfstudio (a few minutes) ---------------------------
# Pin Kaggle's preinstalled torch so pip can't swap it for a CUDA-mismatched build.
from importlib.metadata import version, PackageNotFoundError
pins = []
for pkg in ("torch", "torchvision", "torchaudio"):
    try:
        pins.append(f"{pkg}=={version(pkg)}")
    except PackageNotFoundError:
        pass
open(f"{WORK}/constraints.txt", "w").write("\n".join(pins) + "\n")
!apt-get update -qq && apt-get install -y -qq colmap > /dev/null
!pip install -q nerfstudio plyfile -c "{WORK}/constraints.txt"
!colmap -h | head -n 1
print("torch pinned:", pins)

# PyTorch >= 2.6 defaults torch.load(weights_only=True), which can't read
# Nerfstudio checkpoints, so ns-export fails. Our own checkpoint is trusted.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
# Build gsplat's CUDA kernels one job at a time (parallel builds can OOM on Kaggle).
os.environ["MAX_JOBS"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"  # T4; use 6.0 on a P100

In [ ]:
# ---- Step 1: camera poses with COLMAP (slowest CPU step) --------------------
# apt's COLMAP is CPU-only, hence --no-gpu. Sequential matching compares each
# frame with its neighbours (plus loop detection), ideal for video.
import json
os.environ["QT_QPA_PLATFORM"] = "offscreen"
!ns-process-data images --data "{FRAMES}" --output-dir "{PROC}" --no-gpu --matching-method {MATCHING} --num-downscales 0

frames = len(json.load(open(f"{PROC}/transforms.json"))["frames"])
print(f"\nCOLMAP registered {frames} / {len(picked)} frames")
if frames < 0.6 * len(picked):
    print("WARNING: many frames failed to register (low overlap / parallax, or mixed lenses). "
          "Results may have holes. Try MATCHING='exhaustive', or keep only one clip.")

In [ ]:
# ---- Step 2: train the splat -----------------------------------------------
# First run compiles gsplat's CUDA kernels (can take ~10 min before step 0).
!ns-train splatfacto --data "{PROC}" --output-dir "{OUT}" --vis tensorboard --max-num-iterations {ITERATIONS}

In [ ]:
# ---- Step 3: export .ply ---------------------------------------------------
import glob
cfg = sorted(glob.glob(f"{OUT}/**/config.yml", recursive=True), key=os.path.getmtime)[-1]
print("config:", cfg)
!ns-export gaussian-splat --load-config "{cfg}" --output-dir "{EXPORT}"
PLY = glob.glob(f"{EXPORT}/*.ply")[0]
print("ply:", PLY, f"{os.path.getsize(PLY) / 1e6:.1f} MB")

In [ ]:
# ---- Step 4: .ply -> .splat (antimatter15 format the app loads) -------------
# 32 bytes/splat: position f32x3, scale f32x3, rgba u8x4, rotation quat u8x4.
# The mobile build keeps the MAX_SPLATS most useful splats: near the subject,
# not huge (big splats are mostly sky/background floaters), weighted by opacity.
import numpy as np
from plyfile import PlyData

SH_C0 = 0.28209479177387814
KEEP_RADIUS = 1.6   # scene units around the dense centre (Nerfstudio scenes are ~unit scale)
MAX_SCALE = 0.08    # drop splats larger than this on any axis (mobile build only)

def ply_to_splat(ply_path, out_path, max_splats=None):
    v = PlyData.read(ply_path)["vertex"].data
    xyz = np.stack([v["x"], v["y"], v["z"]], 1).astype(np.float32)
    scale = np.exp(np.stack([v["scale_0"], v["scale_1"], v["scale_2"]], 1)).astype(np.float32)
    rot = np.stack([v["rot_0"], v["rot_1"], v["rot_2"], v["rot_3"]], 1).astype(np.float32)
    rot /= np.linalg.norm(rot, axis=1, keepdims=True) + 1e-12
    alpha = 1 / (1 + np.exp(-v["opacity"].astype(np.float32)))
    rgb = 0.5 + SH_C0 * np.stack([v["f_dc_0"], v["f_dc_1"], v["f_dc_2"]], 1)

    ok = np.isfinite(xyz).all(1) & np.isfinite(scale).all(1) & np.isfinite(rot).all(1)
    if max_splats:
        centre = np.median(xyz[ok & (alpha > 0.5) & (scale.max(1) < 0.05)], 0)
        ok &= (np.linalg.norm(xyz - centre, axis=1) < KEEP_RADIUS) & (scale.max(1) < MAX_SCALE) & (alpha > 0.1)
    importance = np.where(ok, alpha * np.cbrt(scale.prod(1)), -1)
    order = np.argsort(-importance)[: int(ok.sum())]
    if max_splats:
        order = order[:max_splats]

    buf = np.zeros(len(order), dtype=[("p", "<f4", 3), ("s", "<f4", 3), ("c", "u1", 4), ("r", "u1", 4)])
    buf["p"] = xyz[order]
    buf["s"] = scale[order]
    rgba = np.concatenate([rgb[order], alpha[order, None]], 1)
    buf["c"] = (np.clip(rgba, 0, 1) * 255).round().astype(np.uint8)
    buf["r"] = np.clip(rot[order] * 128 + 128, 0, 255).astype(np.uint8)
    buf.tofile(out_path)
    print(f"{out_path}: {len(order):,} / {len(v):,} splats, {os.path.getsize(out_path) / 1e6:.1f} MB")

ply_to_splat(PLY, f"{WORK}/{SITE}.splat", MAX_SPLATS)   # mobile build -> public/models/
ply_to_splat(PLY, f"{WORK}/{SITE}-full.splat")           # full quality, for the archive / Ultra

In [ ]:
# ---- Step 5: package outputs -----------------------------------------------
# Files in /kaggle/working appear in the Output panel (download from there).
# proc.tar.gz = cached COLMAP poses: upload it as a Dataset to skip Step 1 next time.
shutil.copy(PLY, f"{WORK}/{SITE}.ply")
!tar -czf "{WORK}/proc.tar.gz" -C "{WORK}" proc
!rm -rf "{OUT}" "{EXPORT}" "{PROC}" "{FRAMES}"
!ls -lh "{WORK}"

## Next: into the app

1. Download `<SITE>.splat` from **Output**.
2. (Optional) open `<SITE>.ply` in **SuperSplat** (superspl.at/editor) to delete floaters/sky, then export `.splat`.
3. Copy to `public/models/<SITE>.splat` and toggle **◍ Photoreal** in the app.
4. Preview with `/splat-preview.html?src=/models/<SITE>.splat&rx=90`: Nerfstudio exports are Z-up, so `rx=90` stands them upright (see `docs/CAPTURE.md` §4).
5. Test on a real low-end Android over 4G. If it's slow, lower `MAX_SPLATS` and re-run only Step 4.